# OpenFold3 ddG Stand

Ноутбук для panel-based ddG-сканирования одной белковой цепи во все 19 аминокислотных замещений на каждой выбранной позиции.

Что делает workflow:
- принимает `PDB ID` двухбелкового комплекса;
- выбирает одну мутируемую белковую цепь;
- строит WT + 19 мутантов на каждую позицию;
- сохраняет прогресс в `state.sqlite` и умеет продолжать после сброса;
- считает консенсусный ranking по 5 ddG-методам из testbench;
- позволяет визуально сравнить WT complex, FoldX local mutant и OpenFold mutant complex.

Базовый безопасный пример по умолчанию: `1BRS` / barnase-barstar, мутируемая цепь `D`.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_ROOT.parent
HELPERS_DIR = NOTEBOOK_ROOT / 'helpers'
OPENFOLD_REPO_DIR = Path(os.environ.get('OPENFOLD_REPO_DIR', str(REPO_ROOT / 'openfold-3'))).resolve()
SAFE_TRITON_CACHE_DIR = Path(os.environ.get('TRITON_CACHE_DIR', '/tmp/openfold3_triton_cache')).resolve()

os.environ.setdefault('OPENFOLD_DISABLE_OPTIONAL_DEEPSPEED_IMPORT', '1')
os.environ.setdefault('TRITON_CACHE_DIR', str(SAFE_TRITON_CACHE_DIR))
os.environ.setdefault('DS_BUILD_AIO', '0')
os.environ.setdefault('AIO', '0')
SAFE_TRITON_CACHE_DIR.mkdir(parents=True, exist_ok=True)

for path in (NOTEBOOK_ROOT, HELPERS_DIR, OPENFOLD_REPO_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from of_notebook_lib import RuntimeConfig, preview_molecules
from of_notebook_lib.ddg_panel import (
    DEFAULT_SAFE_PPI_TARGET,
    build_run_name,
    load_panel_visual_rows,
    preview_panel_input,
    render_info_card,
    render_panel_structure_comparison_html,
)
from of_notebook_lib.query_builders import build_single_query_payload
from openfold3.panel_stand import PanelDdgStandRunner, PanelStandConfig
from openfold3.panel_summary import summarize_panel_state_db, write_panel_summary_outputs

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 200)


## 1. Runtime

Эта ячейка только поднимает безопасное окружение для `run_openfold` и panel-стенда.

In [ ]:
runtime = RuntimeConfig(
    project_dir=REPO_ROOT,
    openfold_repo_dir=OPENFOLD_REPO_DIR,
    openfold_prefix=Path(os.environ.get('OPENFOLD_PREFIX', str(REPO_ROOT / '.venv'))),
    results_dir=REPO_ROOT / 'results',
    msa_cache_dir=REPO_ROOT / 'msa_cache' / 'colabfold_msas',
    triton_cache_dir=SAFE_TRITON_CACHE_DIR,
    fixed_msa_tmp_dir=REPO_ROOT / '.runtime' / 'of3_colabfold_msas',
    use_fused_attention=False,
    use_deepspeed=False,
)

runtime_env = runtime.build_env()
for key, value in runtime_env.items():
    os.environ[key] = value

display(HTML(render_info_card(
    'Runtime',
    [
        ('project_dir', runtime.project_dir),
        ('openfold_repo_dir', runtime.openfold_repo_dir),
        ('openfold_runner', runtime.openfold_runner),
        ('TRITON_CACHE_DIR', os.environ.get('TRITON_CACHE_DIR')),
        ('DeepSpeed optional import disabled', os.environ.get('OPENFOLD_DISABLE_OPTIONAL_DEEPSPEED_IMPORT')),
    ],
    accent='#1c7ed6',
)))


## 2. Основные параметры

Пользовательские вентильные параметры. Здесь задаётся только то, что реально нужно менять между запусками.

In [ ]:
pdb_id = DEFAULT_SAFE_PPI_TARGET['pdb_id']
target_id = DEFAULT_SAFE_PPI_TARGET['target_id']
mutable_chain_id = DEFAULT_SAFE_PPI_TARGET['mutable_chain_id']

# positions_mode: 'explicit_list' or 'all_chain_positions'
positions_mode = 'explicit_list'
positions_text = '35-45'

experiment_name = build_run_name(pdb_id=pdb_id, mutable_chain_id=mutable_chain_id)
output_parent = REPO_ROOT / 'results' / 'ddg_panels'
pdb_cache_dir = NOTEBOOK_ROOT / '.pdb_cache'

# Model and runtime knobs
runner_yaml = None
inference_ckpt_path = None
inference_ckpt_name = None
msa_computation_settings_yaml = None
num_diffusion_samples = 2
num_model_seeds = 1
msa_panel_workers = 1
analysis_workers = 4
reuse_wt_msa_for_mutants = True
predict_strategy = 'adaptive'
predict_panel_chunk_size = 8
cleanup_intermediates = True
enable_profiling = True
profiling_sample_interval_seconds = 1.0


## 3. Предпросмотр входа

Notebook скачивает `PDB ID`, извлекает белковые цепи, проверяет длину целевой цепи и показывает, сколько именно мутантов будет создано.

In [ ]:
resolved_molecules, pdb_preview_df, panel_preview_df, panel_summary_input, positions = preview_panel_input(
    pdb_id=pdb_id,
    mutable_chain_id=mutable_chain_id,
    positions_mode=positions_mode,
    positions_text=positions_text,
    pdb_cache_dir=pdb_cache_dir,
)

display(HTML(render_info_card(
    'План панели',
    [
        ('pdb_id', panel_summary_input['pdb_id']),
        ('mutable_chain_id', panel_summary_input['mutable_chain_id']),
        ('sequence_length', panel_summary_input['sequence_length']),
        ('positions_count', panel_summary_input['positions_count']),
        ('planned_mutants', panel_summary_input['planned_mutants']),
    ],
    accent='#2f9e44',
)))

display(pdb_preview_df)
display(preview_molecules(resolved_molecules))
display(panel_preview_df.head(50))


## 4. Подготовка run-root

Эта ячейка создаёт `run_root`, пишет WT query и собирает конфиг для staged-runner.

Важно: downstream-мутантные структуры обычно сохраняются как `.cif`, но viewer умеет и `.pdb`, и `.cif`.

In [ ]:
run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_root = output_parent / f'{experiment_name}_{run_stamp}'
run_root.mkdir(parents=True, exist_ok=True)

wt_query_path = run_root / 'wt_query.json'
wt_query_payload = build_single_query_payload(f'{target_id}_WT', resolved_molecules)
wt_query_path.write_text(json.dumps(wt_query_payload, indent=2), encoding='utf-8')

panel_config = PanelStandConfig(
    target_id=target_id,
    wt_query_json=wt_query_path,
    output_root=run_root,
    mutable_chain_id=mutable_chain_id,
    positions=positions,
    runner_yaml=None if runner_yaml is None else Path(runner_yaml),
    inference_ckpt_path=None if inference_ckpt_path is None else Path(inference_ckpt_path),
    inference_ckpt_name=inference_ckpt_name,
    msa_computation_settings_yaml=None if msa_computation_settings_yaml is None else Path(msa_computation_settings_yaml),
    num_diffusion_samples=num_diffusion_samples,
    num_model_seeds=num_model_seeds,
    msa_panel_workers=msa_panel_workers,
    analysis_workers=analysis_workers,
    reuse_wt_msa_for_mutants=reuse_wt_msa_for_mutants,
    predict_strategy=predict_strategy,
    predict_panel_chunk_size=predict_panel_chunk_size,
    cleanup_intermediates=cleanup_intermediates,
    enable_profiling=enable_profiling,
    profiling_sample_interval_seconds=profiling_sample_interval_seconds,
)

plan_payload = {
    'run_root': str(run_root),
    'wt_query_path': str(wt_query_path),
    'pdb_id': pdb_id,
    'target_id': target_id,
    'mutable_chain_id': mutable_chain_id,
    'positions': list(positions),
    'planned_mutants': panel_summary_input['planned_mutants'],
    'predict_strategy': predict_strategy,
}
(run_root / 'plan.json').write_text(json.dumps(plan_payload, indent=2), encoding='utf-8')

display(HTML(render_info_card(
    'Run root prepared',
    [
        ('run_root', run_root),
        ('wt_query_path', wt_query_path),
        ('state_db', run_root / 'state.sqlite'),
        ('planned_mutants', panel_summary_input['planned_mutants']),
    ],
    accent='#7048e8',
)))


## 5. CPU prepare

Первая стадия: подготовка WT query, `align-msa-server`, reuse WT MSA для мутантов и создание `state.sqlite`.

Эту ячейку можно запускать отдельно, если нужен перенос `run_root` на другой сервер.

In [ ]:
prepare_runner = PanelDdgStandRunner(panel_config)
prepare_payload = prepare_runner.prepare_inputs()

display(HTML(render_info_card(
    'CPU prepare finished',
    [
        ('run_root', prepare_payload['output_root']),
        ('state_db', prepare_payload['state_db']),
        ('panel_count', prepare_payload['panel_count']),
        ('mutant_job_count', prepare_payload['mutant_job_count']),
        ('run_mode', prepare_payload['run_mode']),
    ],
    accent='#1c7ed6',
)))


## 6. GPU resume

Вторая стадия: `predict + analysis`. Если процесс прервался, runner продолжит по `state.sqlite`.

Если запускаете не в той же сессии, просто присвойте `resume_run_root` существующий путь.

In [ ]:
resume_run_root = run_root
# resume_run_root = Path('/abs/path/to/existing_run_root')

resume_run_root = Path(resume_run_root)
resume_config = PanelStandConfig(
    target_id=target_id,
    wt_query_json=resume_run_root / 'wt_query.json',
    output_root=resume_run_root,
    mutable_chain_id=mutable_chain_id,
    positions=positions,
    runner_yaml=None if runner_yaml is None else Path(runner_yaml),
    inference_ckpt_path=None if inference_ckpt_path is None else Path(inference_ckpt_path),
    inference_ckpt_name=inference_ckpt_name,
    msa_computation_settings_yaml=None if msa_computation_settings_yaml is None else Path(msa_computation_settings_yaml),
    num_diffusion_samples=num_diffusion_samples,
    num_model_seeds=num_model_seeds,
    msa_panel_workers=msa_panel_workers,
    analysis_workers=analysis_workers,
    reuse_wt_msa_for_mutants=reuse_wt_msa_for_mutants,
    predict_strategy=predict_strategy,
    predict_panel_chunk_size=predict_panel_chunk_size,
    cleanup_intermediates=cleanup_intermediates,
    enable_profiling=enable_profiling,
    profiling_sample_interval_seconds=profiling_sample_interval_seconds,
)

resume_runner = PanelDdgStandRunner(resume_config)
run_payload = resume_runner.run_predict_and_analysis()
panel_summary = summarize_panel_state_db(Path(run_payload['state_db']))
summary_outputs = write_panel_summary_outputs(panel_summary, Path(run_payload['output_root']) / 'summary_exports')

display(HTML(render_info_card(
    'GPU resume finished',
    [
        ('run_root', run_payload['output_root']),
        ('state_db', run_payload['state_db']),
        ('summary_json', summary_outputs['summary_json']),
        ('rows_csv', summary_outputs['rows_csv']),
        ('top_consensus_count', len(panel_summary.top_consensus)),
    ],
    accent='#2f9e44',
)))

display(pd.DataFrame(panel_summary.top_consensus).head(20))


## 7. Визуальное сравнение структур

Эта секция показывает три представления для выбранного мутанта:
- WT complex;
- FoldX local mutant;
- OpenFold mutant complex.

Если `ipywidgets` нет в kernel, можно просто взять первый доступный кейс и показать его статически.

In [ ]:
visual_rows = load_panel_visual_rows(Path(run_payload['state_db']))
print(f'visual rows: {len(visual_rows)}')

try:
    import ipywidgets as widgets

    options = [(f"{row.chain_id}:{row.from_residue}{row.position_1based}{row.to_residue}", row.job_id) for row in visual_rows]
    picker = widgets.Dropdown(options=options, description='Mutant')
    output = widgets.Output()

    def refresh(*_args):
        selected = next(row for row in visual_rows if row.job_id == picker.value)
        with output:
            output.clear_output()
            display(HTML(render_panel_structure_comparison_html(selected)))

    picker.observe(refresh, names='value')
    refresh()
    display(widgets.VBox([picker, output]))
except Exception:
    if visual_rows:
        display(HTML(render_panel_structure_comparison_html(visual_rows[0])))
    else:
        display(Markdown('В `state.sqlite` пока нет доступных структур для viewer.'))
